In [1]:
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from google.cloud import bigquery, storage

import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType, DateType

gcs_connector_path = '../../config/gcs-connector-hadoop3-latest.jar'
bigquery_connector_path = '../../config/spark-bigquery-with-dependencies_2.12-0.35.0.jar'

load_dotenv()

PostSchema = StructType([
    StructField('Date', DateType(), True),
    StructField('Time', StringType(), True),
    StructField('City', StringType(), True),
    StructField('Location', StringType(), True),
    StructField('Latitude', DoubleType(), True),
    StructField('Longitude', DoubleType(), True),
    StructField('High_Accuracy', DoubleType(), True),
    StructField('Direction', StringType(), True),
    StructField('Type', StringType(), True),
    StructField('Lanes_Blocked', IntegerType(), True),
    StructField('Involved', StringType(), True),
    StructField('Tweet', StringType(), True),
    StructField('Source', StringType(), True),
])

RawSchema = StructType([
    StructField('content', StringType(), True),
    StructField('tweetlinkid', StringType(), True),
    StructField('created_at', DateType(), True),
])

PartialPostSchema = StructType([
    StructField('Date', DateType(), True),
    StructField('Time', StringType(), True),
    StructField('Location', StringType(), True),
    StructField('Direction', StringType(), True),
    StructField('Type', StringType(), True),
    StructField('Lanes_Blocked', IntegerType(), True),
    StructField('Involved', StringType(), True),
    StructField('Tweet', StringType(), True),
    StructField('Source', StringType(), True),
])

In [2]:
import os
import requests
import re
from google.cloud import bigquery
from dotenv import load_dotenv
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, DoubleType, StringType

load_dotenv()
KEY = os.getenv('AZURE_KEY')
CLIENT_ID = os.getenv('CLIENT_ID')
API_VERSION = os.getenv('API_VERSION')

LocationDetailSchema = StructType([
    StructField('City', StringType(), True),
    StructField('Location', StringType(), True),
    StructField('Latitude', DoubleType(), True),
    StructField('Longitude', DoubleType(), True),
    StructField('High_Accuracy', DoubleType(), True)
])

accuracy_map = {
    "High": 1,
    "Medium": 0.5,
    "Low": 0
}

def get_locations_from_bq(df_locations, raw_locations):
    return df_locations.join(raw_locations, on="Location", how='right')

def get_missing_locations(enriched_df):
    # Get null values in City
    missing_df = enriched_df.filter(F.col('City').isNull())

    # Get unique values 
    missing_locations_df = missing_df.select("location").distinct()

    # Convert to array
    missing_locations = [row.location for row in missing_locations_df.collect()]

    return missing_locations


def update_locations_bq(spark, new_locations_df, table_id):
    spark.conf.set('temporaryGcsBucket', 'tempresolvedlocation')

    new_locations_df.write \
                    .format("bigquery") \
                    .option('table', table_id) \
                    .mode("append") \
                    .save()

def get_geocode(location):
    url = "https://atlas.microsoft.com/geocode"

    params = {
        "api-version": "2026-01-01",
        "addressLine": location,
        "adminDistrict": "Metro Manila",
        "countryRegion": "PH",
        "top": 1
    }

    headers = {
        "Accept-Language": "en-US",
        "x-ms-client-id": CLIENT_ID,
        "subscription-key": KEY
    }

    response = requests.get(url=url, params=params, headers=headers)
    data = response.json()

    longitude = data['features'][0]['geometry']['coordinates'][0]
    latitude = data['features'][0]['geometry']['coordinates'][1]
    city = data['features'][0]['properties']['address']['locality']
    accuracy = accuracy_map.get(data['features'][0]['properties']['confidence'], 0)

    return (latitude, longitude, city, accuracy)

def get_batch_geocode(spark, locations):
    body = create_batch_items(locations)

    url = 'https://atlas.microsoft.com/geocode:batch'

    params = {
        "api-version": "2026-01-01",
    }
    
    headers = {
        "Accept-Language": "en-US",
        "x-ms-client-id": CLIENT_ID,
        "subscription-key": KEY
    }

    response = requests.post(url=url, params=params, headers=headers, json=body)
    data = response.json()
    items = data['batchItems']

    details = []
    for item in items:
        feature = item.get('features', [None])[0] if item.get('features') else None

        if feature:
            geometry = feature.get('geometry', None)
            coordinates = geometry.get('coordinates', [None, None])
            longitude = float(coordinates[0])
            latitude = float(coordinates[1])

            properties = feature.get('properties', None)
            address = properties.get('address', None)
            city = address.get('locality', None)

            confidence = properties.get('confidence', None)
            accuracy = float(accuracy_map.get(confidence, 0))
        else:
            longitude = latitude = city = None
            accuracy = 0.0
        
        if city in ["Pasay", "Pasig", "Makati"]:
            city = city.strip() + " City"
        
        if city == "Kalookan City":
            city = "Caloocan City"

        pattern = re.compile(r'Para.*aque')
        if re.fullmatch(pattern, city):
            city = "Paranaque"
        
        details.append({
            "Longitude": longitude,
            "Latitude": latitude,
            "City": city,
            "High_Accuracy": accuracy
        })
    
    output_details = list(zip(locations, details))
    flattened_data = [(detail["City"], loc, detail["Latitude"], detail["Longitude"], detail["High_Accuracy"]) for loc, detail in output_details]

    # Create DataFrame
    df = spark.createDataFrame(flattened_data, LocationDetailSchema)
    return df


def create_batch_items(locations):

    batch_items = []

    for loc in locations:
        batch_items.append({
            "addressLine": loc,
            "adminDistrict": "Metro Manila",
            "countryRegion": "PH",
            "top": 1
        })

    return {"batchItems": batch_items}

# def get_reverse_geocode(longitude, latitude):

#     coordinates = str(longitude) + "," + str(latitude)
#     url = 'https://atlas.microsoft.com/reverseGeocode'

#     params = {
#         "api-version": "2026-01-01",
#         "coordinates": coordinates
#     }

#     headers = {
#         "Accept-Language": "en-US",
#         "x-ms-client-id": CLIENT_ID,
#         "subscription-key": KEY
#     }

#     response = requests.get(url=url, params=params, headers=headers)
#     data = response.json()
#     city = data['features'][0]['properties']['address']['locality']

#     return city

In [35]:
import re
import logging
from datetime import datetime, timedelta

word_2_num = {
    'ZERO': 0,
    'ONE': 1,
    'TWO': 2,
    'THREE': 3,
    'FOUR': 4,
    'FIVE': 5,
    'SIX': 6,
    'SEVEN': 7,
    'EIGHT': 8,
    'NINE': 9,
    'TEN': 10
}

directions = ['NB', 'SB', 'EB', 'WB']

def strip_direction(text):
    """
    Remove direction indicators from text.
    """
    for dir in [' NB', ' EB', ' SB', ' WB', ' NB ', ' EB ', ' SB ', ' WB ']:
        text = text.replace(dir, ' ')
    return text.strip()


def get_time(tweet_text):
    """
    Extract time from MMDA tweet.
    """
    tweet_text = tweet_text.upper().replace(';', ':')
    pattern = re.compile(r'\d+:\d\d[\s(AM|PM)]+')
    matches = pattern.finditer(tweet_text)
    logging.info('get_time(): Raw input {}'.format(tweet_text))

    for match in matches:
        tweet_text = match.group(0)
        logging.info('get_time(): RegEx Match {}'.format(tweet_text))
        tweet_text = tweet_text.replace('.', '')

    if len(tweet_text) > 10:
        return ''

    numList = ['1','2','3','4','5','6','7','8','9','0',':']
    pattern = re.compile(r'[0-9]\s[(AM|PM)]')
    matches = pattern.finditer(tweet_text)
    timeCheck = ''
    for match in matches:
        timeCheck = match.group(0)
        logging.info('get_time(): TimeCheck Var {}'.format(timeCheck))

    pattern = re.compile(r'(AM|PM)')
    matches = pattern.finditer(tweet_text)
    timeDay = ''
    for match in matches:
        timeDay = match.group(0)
        logging.info('get_time(): timeDay Var {}'.format(timeDay))

    if len(timeCheck) == 0 and ('AM' in tweet_text or 'PM' in tweet_text):
        stringFix = ''.join([x for x in tweet_text if x in numList])
        tweet_text = stringFix + ' ' + timeDay
        logging.info('get_time(): Cleaned Output {}'.format(tweet_text))

    return tweet_text


def get_lanes_blocked(tweet_text):
    """
    Extract number of lanes blocked from tweet.
    """
    pattern = re.compile(r'((\d\s*)|(\w*\s*))(LANE|LANES)')
    matches = pattern.finditer(tweet_text)
    logging.info('get_lanes_blocked(): Raw input {}'.format(tweet_text))

    tweet_lanes = ''
    for match in matches:
        tweet_text = match.group(0)
        logging.info('get_lanes_blocked(): RegEx Match {}'.format(tweet_text))
        tweet_lanes = tweet_text.split(' ')[0]

    if not tweet_lanes.strip().isdigit():
        tweet_lanes = word_2_num.get(tweet_lanes.strip(), '')

    logging.info('get_lanes_blocked(): Cleaned output {}'.format(tweet_lanes))
    return tweet_lanes


def get_inc_type(tweet_text):
    """
    Extract incident type from tweet.
    """
    parsed_incident_type = False
    tweet_text = tweet_text.upper()
    pattern = re.compile(r'MMDA ALERT: .* AT ')
    matches = pattern.finditer(tweet_text)
    logging.info('get_inc_type(): Raw input {}'.format(tweet_text))

    for match in matches:
        tweet_text = match.group(0)
        logging.info('get_inc_type(): RegEx Match {}'.format(tweet_text))
        tweetType = tweet_text.replace('MMDA ALERT: ', '').replace(' AT ', '')
        logging.info('get_inc_type(): Cleaned output {}'.format(tweetType))
        parsed_incident_type = True

    if not parsed_incident_type:
        tweetType = ''
        logging.info('get_inc_type(): Empty output')

    return tweetType


def get_direction(tweet_text):
    """
    Extract direction from tweet.
    """
    parsed_direction = False
    pattern = re.compile(r'( SB | NB | WB | EB | SB| NB| WB| EB)')
    matches = pattern.finditer(tweet_text)
    for match in matches:
        tweetDirection = match.group(0).strip()
        parsed_direction = True

    if not parsed_direction:
        tweetDirection = ''

    return tweetDirection


def get_location(tweet_text, strip_dir=True):
    """
    Extract location from tweet.
    """
    pattern = re.compile(r' AT\s[a-zA-Z\Ñ\'\.\,\-0-9\/\s]+(AS OF)')
    matches = pattern.finditer(tweet_text)
    tweet_location = ''
    tweet_location_final = ''

    for match in matches:
        tweet_location = match.group(0)
        logging.info('get_location(): RegEx Match {}'.format(tweet_location))

        if any(direction in tweet_location for direction in directions):
            pattern2 = re.compile(r'AT\s+(.*?)(?:\s(NB|SB|EB|WB)\b|$)')
            matches2 = pattern2.finditer(tweet_location)
            for match2 in matches2:
                tweet_location_final = match2.group(1)
        else:    
            tweet_location = tweet_location.replace('AT ', '').replace(' AS OF', '').strip()
            tweet_location_final = location_string_clean(tweet_location)
            if strip_dir:
                tweet_location_final = strip_direction(tweet_location_final)

        if "INVOLVING" in tweet_location_final:
            pattern3 = re.compile(r'(.*)\s+INVOLVING')
            matches3 = pattern3.finditer(tweet_location_final)
            for match3 in matches3:
                tweet_location_final = match3.group(1)

    logging.info('get_location(): Cleaned Location {}'.format(tweet_location_final))
    return tweet_location_final.replace('Ñ', 'N').strip()


def get_participants(tweet_text):
    """
    Extract participants from tweet.
    """
    logging.info('get_participants(): Raw input {}'.format(tweet_text))
    if ' INVOLVING' in tweet_text:
        tweet_participant = tweet_text.split(' INVOLVING')[1].split('AS OF')[0].strip()
        logging.info('get_participants(): Cleaned output {}'.format(tweet_participant))
    else:
        tweet_participant = ''
    return tweet_participant


def get_rally_location(tweet_text):
    """
    Extract rally location from tweet.
    """
    parsed_rally_location = False
    pattern = re.compile(r' AT\s*(.*)\s*MORE OR')
    matches = pattern.finditer(tweet_text)
    logging.info('get_rally_location(): Raw input {}'.format(tweet_text))

    for match in matches:
        tweetLocation = match.group(0).replace(' AT ', '').replace(' MORE OR', '')
        tweetLocation = strip_direction(tweetLocation)
        logging.info('get_rally_location(): Cleaned string {}'.format(tweetLocation))

        if any(direction in tweetLocation for direction in directions):
            pattern2 = re.compile(r'AT\s+(.*?)(?:\s(NB|SB|EB|WB)\b|$)')
            matches2 = pattern2.finditer(tweetLocation)
            for match2 in matches2:
                tweetLocation = match2.group(1)
            logging.info('get_location(): Cleaned Location {}'.format(tweetLocation))

        parsed_rally_location = True

    if not parsed_rally_location:
        tweetLocation = ''
        logging.info('get_rally_location(): Empty match')

    return tweetLocation


def get_rally_participants(tweet_text):
    """
    Extract rally participants from tweet.
    """
    parsed_rally_participant = False
    pattern = re.compile(r'MORE OR LESS \d+ PAX')
    matches = pattern.finditer(tweet_text)
    logging.info('get_rally_participants(): Raw input {}'.format(tweet_text))

    for match in matches:
        tweet_participant = match.group(0).replace('MORE OR LESS ', '')
        logging.info('get_rally_participants(): Cleaned output {}'.format(tweet_participant))
        parsed_rally_participant = True

    if not parsed_rally_participant:
        tweet_participant = ''

    return tweet_participant


def get_stalled_participants(tweet_text):
    """
    Extract stalled participants from tweet.
    """
    parsed_stalled_participants = False
    logging.info('get_stalled_participants(): Raw input {}'.format(tweet_text))
    pattern = re.compile(r'STALLED [A-Z0-9\-\s]+DUE')
    matches = pattern.finditer(tweet_text)

    for match in matches:
        tweet_text = match.group(0).replace('STALLED ', '').replace(' DUE', '').strip()
        tweet_participants = tweet_text
        logging.info('get_stalled_participants(): Cleaned String {}'.format(tweet_participants))
        parsed_stalled_participants = True

    if not parsed_stalled_participants:
        tweet_participants = ''
        logging.info('get_stalled_participants(): Empty Match')

    return tweet_participants

def get_location_details_raw(location):
    latitude, longitude, city, accuracy = query_location(location)

    return (latitude, longitude, city, accuracy)


def post_parser(content, created_at, source):
    time = get_time(content)
    date = created_at
    lanes_blocked = get_lanes_blocked(content)
    inc_type = get_inc_type(content)
    direction = get_direction(content)

    if 'RALLY' in content:
        location = get_rally_location(content)
        participants = get_rally_participants(content)
    elif 'STALLED' in content:
        location = get_location(content)
        participants = get_stalled_participants(content)
    else:
        location = get_location(content)
        participants = get_participants(content)

    return (date, time, location, direction, inc_type, lanes_blocked, participants, content, source)

In [4]:
def load_locations_df(spark, table_id):
    return spark.read.format("bigquery") \
                .option('table', table_id) \
                .load()

def get_current_raw_data(scrape_folder):
    PHT = ZoneInfo("Asia/Manila")
    now = datetime.now(PHT)
    yesterday = now - timedelta(days=1)

    year = yesterday.strftime("%Y")
    month = yesterday.strftime("%m")
    day = yesterday.strftime("%d")

    filename = f"{scrape_folder}/{year}/{month}/scrape_data_{year}{month}{day}.csv"
    return filename

def gcs_file_read(spark, bucket_name, filename):
    df = spark.read \
        .option("header", True) \
        .option("multiline", True) \
        .option("quote", '"') \
        .option("escape", '"') \
        .option("ignoreLeadingWhiteSpace", True) \
        .option("ignoreTrailingWhiteSpace", True) \
        .schema(RawSchema) \
        .csv(f"gs://{bucket_name}/{filename}")
    
    return df

def partial_parse_raw_data(df_raw):
    post_parser_udf = F.udf(post_parser, PartialPostSchema)

    df_temp = df_raw.withColumn("parsed", 
                                post_parser_udf(
                                    F.upper(df_raw['content']),
                                    df_raw['created_at'],
                                    df_raw['tweetlinkid']
                                    )
                                )
    
    return df_temp.select(
        F.col("parsed.Date").alias("Date"),
        F.col("parsed.Time").alias("Time"),
        F.col("parsed.Location").alias("Location"),
        F.col("parsed.Direction").alias("Direction"),
        F.col("parsed.Type").alias("Type"),
        F.col("parsed.Lanes_Blocked").alias("Lanes_Blocked"),
        F.col("parsed.Involved").alias("Participants"),
        F.col("parsed.Tweet").alias("Tweet"),
        F.col("parsed.Source").alias("Source")
    )

In [5]:
spark = SparkSession.builder \
        .master("local[*]") \
        .appName('Transform Stage') \
        .config("spark.jars", f"{gcs_connector_path},{bigquery_connector_path}") \
        .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
        .config("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
        .getOrCreate() 

26/03/29 19:48:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [6]:
gcs_client = storage.Client()
project_id = os.getenv("PROJECT_ID")
dataset = os.getenv("DATASET")
table_id = f"{project_id}:{dataset}.locations"
bucket_name = os.getenv('BUCKET_NAME')
scrape_folder="raw/scrape"

bucket = gcs_client.bucket(bucket_name)

In [7]:
df_locations = load_locations_df(spark, table_id)

In [ ]:
raw_filename = get_current_raw_data(scrape_folder)
is_file_exists = storage.Blob(bucket=bucket, name=raw_filename).exists()

In [36]:
df_raw = gcs_file_read(spark, bucket_name, raw_filename)

In [10]:
df_raw.show(5)

+--------------------+--------------------+----------+
|             content|         tweetlinkid|created_at|
+--------------------+--------------------+----------+
|MMDA ALERT: Road ...|https://x.com/MMD...|2026-03-29|
|MMDA ALERT: Stall...|https://x.com/MMD...|2026-03-29|
|MMDA ALERT: Stall...|https://x.com/MMD...|2026-03-29|
|MMDA ALERT: Road ...|https://x.com/MMD...|2026-03-29|
|MMDA ALERT: Stall...|https://x.com/MMD...|2026-03-29|
+--------------------+--------------------+----------+
only showing top 5 rows



In [37]:
df_partial_parsed = partial_parse_raw_data(df_raw)

In [12]:
df_partial_parsed.show(5, truncate=False)

+----------+-------+-------------------------------+---------+----------------------------------------+-------------+------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------+
|Date      |Time   |Location                       |Direction|Type                                    |Lanes_Blocked|Participants      |Tweet                                                                                                                                                                           |Source                                         |
+----------+-------+-------------------------------+---------+----------------------------------------+-------------+------------------+--------------------------------------------------------------------------------------------------------------------------------------------

In [38]:
df_full_parsed = get_locations_from_bq(df_locations, df_partial_parsed)

In [39]:
df_full_parsed.show(5, truncate=False)

+-------------------------------+-----------+----------------+----------------+-------------+----------+-------+---------+-----------------------------------------+-------------+------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------+
|Location                       |City       |Latitude        |Longitude       |High_Accuracy|Date      |Time   |Direction|Type                                     |Lanes_Blocked|Participants      |Tweet                                                                                                                                                                           |Source                                         |
+-------------------------------+-----------+----------------+----------------+-------------+----------+-------+---------+--------------------------------

In [14]:
missing_locations = get_missing_locations(df_full_parsed)

In [15]:
missing_locations

[]

In [16]:
len(missing_locations)

0

In [18]:
if len(missing_locations) != 0:
    print('TRUE')
else:
    print('FALSE')


FALSE


In [40]:
df_full_parsed = df_full_parsed.withColumnRenamed('Participants', 'Involved')

In [41]:
df_final = df_full_parsed.select('Date', 'Time', 'City', 'Location', 'Latitude', 'Longitude', 'High_Accuracy', 'Direction', 'Type', 'Lanes_Blocked', 'Involved', 'Tweet', 'Source')

In [42]:
df_final.show(5, truncate=False)

+----------+-------+-----------+-------------------------------+----------------+----------------+-------------+---------+-----------------------------------------+-------------+------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------+
|Date      |Time   |City       |Location                       |Latitude        |Longitude       |High_Accuracy|Direction|Type                                     |Lanes_Blocked|Involved          |Tweet                                                                                                                                                                           |Source                                         |
+----------+-------+-----------+-------------------------------+----------------+----------------+-------------+---------+--------------------------------

In [43]:
def get_current_filename(scrape_folder, clean_folder):
    PHT = ZoneInfo("Asia/Manila")
    now = datetime.now(PHT)
    yesterday = now - timedelta(days=1)

    year = yesterday.strftime("%Y")
    month = yesterday.strftime("%m")
    day = yesterday.strftime("%d")

    raw_filename = f"{scrape_folder}/{year}/{month}/scrape_data_{year}{month}{day}.csv"
    output_filename = f"{clean_folder}/{year}/{month}/scrape_data_{year}{month}{day}.parquet"
    return (raw_filename, output_filename)

In [44]:
load_dotenv()

raw_folder = os.getenv('RAW_FOLDER_NAME')
clean_folder = os.getenv('CLEANED_FOLDER_NAME')

raw_filename, output_filename = get_current_filename(scrape_folder, clean_folder)

In [46]:
bucket_name

'mnl_accident_pipeline_bucket'

In [47]:
f"{bucket_name}/{output_filename}"

'mnl_accident_pipeline_bucket/cleaned/2026/03/scrape_data_20260329.parquet'

In [52]:
df_final.write \
        .mode("overwrite") \
        .partitionBy("Date") \
        .parquet(f"gs://{bucket_name}/cleaned/")
        